# PINN and KANN approaches for shell structures: sampling strategies and experimental analysis

**Paper:** Rodrigues, F.V.B., Alves Macedo, L.F., Pimenta, P.M. (2025). *PINN and KAN approaches for shell structures: sampling strategies and experimental analysis.* Proceedings of the XLVI Ibero-Latin-American Congress on Computational Methods in Engineering (CILAMCE-2025), Vitoria, Brazil.

**Carpeta origen:** `PINNs/1. mecanica de fluidos/PINN_and_KANN_approaches_for_shell_structures_samp.pdf`

## Como se usan las PINNs en este paper

El paper compara una **PINN estandar (MLP)** contra una **KAN-PINN** (red de Kolmogorov-Arnold con activaciones B-spline aprendibles) para resolver el problema mecanico de una **cascara delgada de Naghdi** (5 grados de libertad: 3 desplazamientos $\mathbf{u}$ + 2 rotaciones $\boldsymbol\theta$), usando el benchmark clasico del **techo de Scordelis-Lo** (Eq. 6): una superficie cilindrica de radio $r=0.5$

$$\phi(\xi_1,\xi_2)=\Big\{\xi_1,\ \tfrac12\sin\xi_2,\ \tfrac12\cos\xi_2\Big\}$$

empotrada en los extremos ($\hat u_2=\hat u_3=0$), bajo carga gravitatoria, con $\nu=0$.

**Punto metodologico central:** a diferencia de la mayoria de PINNs (forma fuerte, residuo de la EDP), aqui se usa la **forma debil/energetica** (estilo *Deep Ritz*): la red predice $(\mathbf{u},\boldsymbol\theta)$, se calculan las deformaciones de membrana $e_{\alpha\beta}$, flexion $\kappa_{\alpha\beta}$ y cortante $\gamma_\alpha$ (Eq. 1-3) por diferenciacion automatica, y la red se entrena **minimizando directamente la energia potencial total** (Eq. 4-5):

$$\Pi[\mathbf{u},\boldsymbol\theta]=\frac12\int_\omega\Big[\mathbb{C}^{\alpha\beta\sigma\rho}\big(t\,e_{\alpha\beta}e_{\sigma\rho}+\tfrac{t^3}{12}\kappa_{\alpha\beta}\kappa_{\sigma\rho}\big)+\mathbb{D}^{\alpha\beta}t\kappa\gamma_\alpha\gamma_\beta\Big]dS - W_{ext}$$

sin necesidad de imponer un residuo de EDP en forma fuerte; las condiciones de contorno esenciales se imponen con funciones de prueba suaves (restriccion dura).

Este cuaderno reproduce fielmente la variante **PINN (MLP)** de este framework: la parametrizacion geometrica exacta de la cascara cilindrica (Eq. 6), las deformaciones de membrana/flexion/cortante especializadas a esta geometria cilindrica, la energia potencial (Eq. 4) y su minimizacion directa como funcion de perdida (forma energetica/debil), con restriccion dura en los extremos empotrados.

**Simplificacion declarada:** se usa la especializacion estandar de las deformaciones de Naghdi para una cascara **cilindrica** (coordenadas de longitud de arco), en vez de re-derivar los tensores covariantes generales completos (simbolos de Christoffel, formas fundamentales) letra por letra. Con $\nu=0$ (como en el paper) los tensores constitutivos se desacoplan, lo cual se aprovecha aqui. No se reproduce la variante **KAN** (requeriria implementar capas B-spline desde cero); se documenta donde se sustituiria.

## Repositorio publico de referencia

El PDF no incluye repositorio propio, pero **cita explicitamente como referencia [1]** el trabajo de Bastek & Kochmann, *Physics-Informed Neural Networks for shell structures*, que introduce exactamente este mismo framework (teoria de Naghdi + PINN en forma debil/energetica) que este paper extiende con KANs. Su repositorio oficial:

- **jhbastek/PhysicsInformedShellStructures** &mdash; https://github.com/jhbastek/PhysicsInformedShellStructures

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Geometria del techo de Scordelis-Lo (Eq. 6) y parametros

In [ ]:
r = 0.5
L = 0.5                  # semi-longitud axial, xi1 in [-L, L]
theta0 = np.deg2rad(40)  # angulo central, xi2 in [-theta0/2, theta0/2]... el paper usa [-2pi/9, 2pi/9]
xi2_max = 2 * np.pi / 9

t = 0.05     # relacion espesor/longitud (caso de referencia del paper)
E = 1.0      # modulo elastico (normalizado; el paper no reporta el valor absoluto usado)
nu = 0.0     # Poisson, como en el paper
kappa_shear = 5.0 / 6.0
G = E / (2 * (1 + nu))
g_load = 1.0  # magnitud de carga gravitatoria normalizada

print(f'Con nu=0: modulo membrana/flexion C={E:.3f}, modulo cortante D={G:.3f}')

## 2. Red PINN con restriccion dura en los extremos empotrados ($\hat u_2=\hat u_3=0$ en $\xi_1=\pm L$)

In [ ]:
class ShellPINN(nn.Module):
    """MLP: 3 capas ocultas x 50 neuronas, GELU (Seccion 4 del paper)."""
    def __init__(self, n_hidden=3, n_neurons=50):
        super().__init__()
        layers = [nn.Linear(2, n_neurons), nn.GELU()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.GELU()]
        layers += [nn.Linear(n_neurons, 5)]  # u1, u2, u3, theta1, theta2
        self.net = nn.Sequential(*layers)

    def forward(self, xi1, xi2):
        x = torch.cat([xi1 / L, xi2 / xi2_max], dim=1)
        out = self.net(x)
        u1, u2, u3, th1, th2 = [out[:, i:i+1] for i in range(5)]
        clamp_factor = (L**2 - xi1**2) / L**2  # se anula en xi1 = +-L
        u2 = clamp_factor * u2
        u3 = clamp_factor * u3
        return u1, u2, u3, th1, th2


model = ShellPINN().to(device)


def d_d(f, v):
    return torch.autograd.grad(f, v, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]

## 3. Deformaciones (Eq. 1-3, especializadas a la geometria cilindrica) y energia potencial (Eq. 4)

Se usa la coordenada de arco circunferencial $y=r\xi_2$ para evitar tener que normalizar por el tensor metrico explicitamente.

In [ ]:
def strains(model, xi1, xi2):
    u1, u2, u3, th1, th2 = model(xi1, xi2)

    du1_1 = d_d(u1, xi1)
    du1_2 = d_d(u1, xi2) / r     # d/dy = (1/r) d/dxi2
    du2_1 = d_d(u2, xi1)
    du2_2 = d_d(u2, xi2) / r
    du3_1 = d_d(u3, xi1)
    du3_2 = d_d(u3, xi2) / r
    dth1_1 = d_d(th1, xi1)
    dth1_2 = d_d(th1, xi2) / r
    dth2_1 = d_d(th2, xi1)
    dth2_2 = d_d(th2, xi2) / r

    # Deformacion de membrana (Eq. 1), especializada a cilindro (curvatura b_22 = -1/r)
    e11 = du1_1
    e22 = du2_2 + u3 / r
    e12 = 0.5 * (du1_2 + du2_1)

    # Deformacion de flexion (Eq. 2)
    k11 = dth1_1
    k22 = dth2_2
    k12 = 0.5 * (dth1_2 + dth2_1)

    # Deformacion de cortante (Eq. 3)
    gamma1 = th1 + du3_1
    gamma2 = th2 + du3_2 - u2 / r

    return u3, (e11, e22, e12), (k11, k22, k12), (gamma1, gamma2)


def potential_energy(model, xi1, xi2):
    u3, (e11, e22, e12), (k11, k22, k12), (gamma1, gamma2) = strains(model, xi1, xi2)

    membrane = E * (e11**2 + e22**2 + 2 * e12**2)
    bending = E * (k11**2 + k22**2 + 2 * k12**2)
    shear = kappa_shear * G * (gamma1**2 + gamma2**2)

    energy_density = 0.5 * (t * membrane + (t**3 / 12) * bending) + t * shear
    dS = r  # elemento de superficie (dS = r d(xi1) d(xi2) para el cilindro)
    strain_energy = torch.mean(energy_density) * (2 * L) * (2 * xi2_max) * dS

    external_work = torch.mean(g_load * u3) * (2 * L) * (2 * xi2_max) * dS
    return strain_energy - external_work

## 4. Entrenamiento: minimizacion directa de la energia potencial (forma debil, sin residuo de EDP)

In [ ]:
N_col = 2000
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
history = []

for epoch in range(4000):
    optimizer.zero_grad()
    xi1 = (torch.rand(N_col, 1, device=device) * 2 * L - L).requires_grad_(True)
    xi2 = (torch.rand(N_col, 1, device=device) * 2 * xi2_max - xi2_max).requires_grad_(True)
    energy = potential_energy(model, xi1, xi2)
    energy.backward()
    optimizer.step()
    history.append(energy.item())
    if epoch % 500 == 0:
        print(f'epoch {epoch:5d} | energia potencial = {energy.item():.5e}')

## 5. Resultados: campo de deflexion vertical $\hat u_3$ (cf. Fig. 3, punto de interes en el borde libre)

In [ ]:
n_side = 60
xi1_grid = torch.linspace(-L, L, n_side, device=device)
xi2_grid = torch.linspace(-xi2_max, xi2_max, n_side, device=device)
X1, X2 = torch.meshgrid(xi1_grid, xi2_grid, indexing='ij')
with torch.no_grad():
    _, u2, u3, _, _ = model(X1.reshape(-1, 1), X2.reshape(-1, 1))
u3_grid = u3.cpu().numpy().reshape(n_side, n_side)

plt.figure(figsize=(6, 5))
im = plt.contourf(X1.cpu().numpy(), X2.cpu().numpy(), u3_grid, levels=30, cmap='RdBu_r')
plt.colorbar(im, label='$\\hat u_3$')
plt.xlabel('$\\xi_1$ (eje axial)')
plt.ylabel('$\\xi_2$ (angulo circunferencial)')
plt.title('Deflexion vertical $\\hat u_3$ del techo de Scordelis-Lo')
plt.show()

with torch.no_grad():
    xi1_mid = torch.zeros(1, 1, device=device)
    xi2_edge = torch.full((1, 1), xi2_max, device=device)
    _, _, u3_mid_edge, _, _ = model(xi1_mid, xi2_edge)
print(f'u3 predicho en el punto medio del borde libre: {u3_mid_edge.item():.4f}')
print('(el paper reporta -0.3024 como referencia FEM para t/L=0.05, con su propia calibracion de unidades E y g)')

plt.figure(figsize=(6, 4))
plt.plot(history)
plt.xlabel('Epoca'); plt.ylabel('Energia potencial total')
plt.title('Convergencia de la minimizacion de energia')
plt.grid(alpha=0.3)
plt.show()